# PV-UAD E2 → E3-TDH + **trainable TransReID initialization** — Kaggle multisession

In [1]:
# ============================================================
# 0. SETTINGS — one stateful E2 -> E3 pipeline; normally change nothing
# ============================================================

import time as _time
NOTEBOOK_SESSION_STARTED_UNIX = _time.time()

PIPELINE_NAME = "PVUAD_E2_E3_TRANSREID_INIT_TRAINABLE"
PIPELINE_TAG = "e2_e3_transreid_init_trainable768_b24_v1"
DATASET_VARIANT = "1000"

WHUMARS1000_INPUT = "/kaggle/input/datasets/cpkimhianh/whu-mars-s3clip"
WHUMARS2337_INPUT = "/kaggle/input/datasets/cpkimhianh/whu-mars-2337"

SOURCE_DIR_OVERRIDE = None

# Official MSMT17 full TransReID checkpoint. Leave both as None unless you want
# to point to a manually attached .pth file. PRETRAIN_PATH_OVERRIDE is kept as
# a backward-compatible alias.
TRANSREID_CHECKPOINT_OVERRIDE = None
PRETRAIN_PATH_OVERRIDE = None
RUN_TRANSREID_INIT_SMOKE_TEST = True

# The target UAD architecture intentionally stays identical to the strong
# 768-D baseline. We transplant the person-ReID-trained Transformer weights
# from TransReID, then TRAIN those weights on WHU-MARS. Source SIE, JPM local
# branches and source classifier/BNNeck are not copied. The trained JPM GLOBAL
# branch b1 is mapped into target block 11 + final norm, because in official JPM
# training that is the actual global final Transformer branch.
TRANSREID_TARGET_STRIDE = [16, 16]
TRANSREID_TARGET_EMBED_DIM = 768
USE_SOURCE_SIE = False
USE_SOURCE_JPM_LOCAL_BRANCHES = False

# Optional emergency overrides. Normally leave all three as None.
E2_RESUME_PATH_OVERRIDE = None
E2_COMPLETED_DIR_OVERRIDE = None
E3_RESUME_PATH_OVERRIDE = None

AUTO_RESUME = True
# Never bootstrap this experiment from the old ImageNet-init or frozen-3840D
# E2 checkpoints. Only checkpoints from this exact pipeline tag are eligible.
ALLOW_LEGACY_E2_BOOTSTRAP = False
RUN_TRAINING = True
RUN_FINAL_FLIP_TTA = True
RUN_METADATA_PAIRING_CHECK = True

# Keep the stage boundary explicit: even if E2 finishes early, end the current
# Kaggle version cleanly. E3 begins only after that saved output is attached to
# a new Save & Run All session.
ALWAYS_START_E3_IN_NEW_SESSION = True

# Keep the exact strong UAD recipe. The ONLY intended E2 change is the
# initialization: ImageNet ViT -> pretrained TransReID ViT backbone weights.
E2_EPOCHS = 60
E2_GLOBAL_BATCH = 24
E2_BASE_LR = 0.008

E3_EPOCHS = 25
E3_GLOBAL_BATCH = 24
E3_BASE_LR = 1e-4
E3_HEAD_LR_MULTIPLIER = 5.0

TIR_GLOBAL_WEIGHT = 0.10
TIR_LOCAL_WEIGHT = 0.03
TIR_LOCAL_KEEP_RATIO = 0.50
FEATURE_PRESERVE_WEIGHT = 0.05
SCENARIO_HARD_WEIGHT = 0.15
SCENARIO_HARD_TOPK = 5
SCENARIO_HARD_MARGIN = 0.05
SCENARIO_HARD_TAU = 0.05
SCENARIO_HARD_START_EPOCH = 3

# One Kaggle session lasts at most 12 h. Training receives the same shared
# deadline even if E2 finishes and E3 starts in the same session.
SESSION_HARD_STOP_HOURS = 10.5
SESSION_STOP_RESERVE_MINUTES = 15
CHECKPOINT_EVERY_MINUTES = 30
TIME_CHECK_EVERY_STEPS = 10
MIN_EVAL_REMAINING_MINUTES = 35

# Do not start a new stage/evaluation too close to the session boundary.
MIN_E3_START_REMAINING_MINUTES = 75
MIN_TTA_START_REMAINING_MINUTES = 50

NUM_GPUS = 2
NUM_WORKERS = 4
RANDOM_SEED = 1234

if DATASET_VARIANT != "1000":
    raise ValueError(
        "This score-oriented pipeline is locked to WHU-MARS-1000. "
        "Adapt to 2337 only after the winning recipe is frozen."
    )
if E2_GLOBAL_BATCH != 24 or abs(E2_BASE_LR - 0.008) > 1e-12:
    raise ValueError("E2 must stay at global batch 24 / LR 0.008.")
if E3_GLOBAL_BATCH != 24 or abs(E3_BASE_LR - 1e-4) > 1e-12:
    raise ValueError("E3 must stay at global batch 24 / LR 1e-4.")
if TRANSREID_TARGET_STRIDE != [16, 16]:
    raise ValueError(
        "Default experiment keeps UAD stride [16,16] so initialization is the "
        "only E2 variable. Test stride-12 only as a later ablation."
    )
if TRANSREID_TARGET_EMBED_DIM != 768:
    raise ValueError("This pipeline must keep the original UAD 768-D feature space.")
if USE_SOURCE_SIE or USE_SOURCE_JPM_LOCAL_BRANCHES:
    raise ValueError(
        "Source SIE/JPM local branches are intentionally disabled in this clean "
        "initialization experiment. The trained JPM global b1 branch is still "
        "used to initialize the target final Transformer block."
    )

SESSION_DEADLINE_UNIX = (
    NOTEBOOK_SESSION_STARTED_UNIX + 3600.0 * SESSION_HARD_STOP_HOURS
)
print(
    PIPELINE_NAME,
    {
        "E2_epochs": E2_EPOCHS,
        "E3_epochs": E3_EPOCHS,
        "deadline_hours": SESSION_HARD_STOP_HOURS,
        "init": "MSMT17 TransReID -> trainable UAD ViT",
        "feature_dim": TRANSREID_TARGET_EMBED_DIM,
        "target_stride": TRANSREID_TARGET_STRIDE,
    },
)


PVUAD_E2_E3_TRANSREID_INIT_TRAINABLE {'E2_epochs': 60, 'E3_epochs': 25, 'deadline_hours': 10.5, 'init': 'MSMT17 TransReID -> trainable UAD ViT', 'feature_dim': 768, 'target_stride': [16, 16]}


## 1. Kiểm tra môi trường

In [2]:
import base64
import csv
import hashlib
import importlib.util
import io
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tarfile
import urllib.request
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator and restart the session.")
if torch.cuda.device_count() < NUM_GPUS:
    raise RuntimeError(
        f"This run expects {NUM_GPUS} GPUs, but Kaggle exposes {torch.cuda.device_count()}. "
        "Choose GPU T4 x2 in Notebook options."
    )

for package, pip_name in [("yacs", "yacs==0.1.8"), ("timm", "timm>=0.9,<2"), ("gdown", "gdown>=5,<6")]:
    if importlib.util.find_spec(package) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            check=True,
        )

gpu_rows = []
for index in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(index)
    gpu_rows.append({
        "index": index,
        "name": props.name,
        "memory_GiB": round(props.total_memory / 2**30, 2),
    })
display(pd.DataFrame(gpu_rows))
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)


,index,name,memory_GiB
0,0,Tesla T4,14.56
1,1,Tesla T4,14.56


Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA runtime: 12.8


## 2. Tìm và khóa WHU-MARS-1000

Tự tìm `train/query/test`, xác nhận đúng 30,711 / 2,135 / 31,203 ảnh cho mỗi modality, rồi dùng symlink để không copy dataset.


In [3]:
IMAGE_SPLITS = {"train", "query", "test", "gallery"}
MODALITIES = ("RGB", "IR", "Thermal")

def find_dataset_roots(base_path, max_depth=6):
    base = Path(base_path)
    if not base.exists():
        return []
    candidates = []
    for root, dirs, _files in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.relative_to(base).parts)
        names = {name.lower(): name for name in dirs}
        if {"train", "query", "test"}.issubset(names):
            candidate = root_path
            if all((candidate / "train" / modality).is_dir() for modality in MODALITIES):
                candidates.append(candidate)
            dirs[:] = []
            continue
        if depth >= max_depth or root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
    return sorted(set(candidates), key=lambda path: (len(path.parts), str(path)))

requested_input = WHUMARS1000_INPUT if DATASET_VARIANT == "1000" else WHUMARS2337_INPUT
candidate_inputs = [
    requested_input,
    "/kaggle/input/whu-mars" if DATASET_VARIANT == "1000" else "/kaggle/input/whu-mars-2337",
    "/kaggle/input",
]
roots = []
for base in candidate_inputs:
    roots.extend(find_dataset_roots(base))
    if roots and base != "/kaggle/input":
        break
roots = sorted(set(roots), key=lambda path: (len(path.parts), str(path)))
if not roots:
    raise FileNotFoundError(
        f"Cannot find train/query/test under {requested_input}. Add both WHU datasets as Kaggle inputs."
    )
if len(roots) > 1:
    exact = [path for path in roots if "2337" not in str(path).lower()]
    if DATASET_VARIANT == "1000" and len(exact) == 1:
        DATA_ROOT = exact[0]
    else:
        raise RuntimeError(f"Ambiguous dataset roots: {[str(path) for path in roots]}")
else:
    DATA_ROOT = roots[0]

def jpg_count(path):
    return sum(1 for item in Path(path).iterdir() if item.is_file() and item.suffix.lower() == ".jpg")

coverage_rows = []
for split in ("train", "query", "test"):
    for modality in MODALITIES:
        coverage_rows.append({
            "split": split,
            "modality": modality,
            "images": jpg_count(DATA_ROOT / split / modality),
        })
coverage = pd.DataFrame(coverage_rows)
display(coverage.pivot(index="split", columns="modality", values="images"))

expected_1000 = {
    ("train", "RGB"): 30711, ("train", "IR"): 30711, ("train", "Thermal"): 30711,
    ("query", "RGB"): 2135, ("query", "IR"): 2135, ("query", "Thermal"): 2135,
    ("test", "RGB"): 31203, ("test", "IR"): 31203, ("test", "Thermal"): 31203,
}
observed = {(row.split, row.modality): int(row.images) for row in coverage.itertuples()}
if observed != expected_1000:
    raise RuntimeError(
        "WHU-MARS-1000 count mismatch. Refusing to train on a wrong/incomplete extraction.\n"
        f"Observed: {observed}"
    )

# The paper's 185,742-image total counts train + test. The 6,405 query
# files are a protocol copy/subset and therefore are not added again.

DATA_LINK_PARENT = Path("/kaggle/working/pvuad_data")
DATA_LINK_PARENT.mkdir(parents=True, exist_ok=True)
DATA_LINK = DATA_LINK_PARENT / "WHU-MARS"
if DATA_LINK.is_symlink():
    DATA_LINK.unlink()
elif DATA_LINK.exists():
    raise RuntimeError(
        f"Expected {DATA_LINK} to be a symlink, but it is a real file/folder. "
        "Use a fresh Kaggle session or inspect it manually."
    )
DATA_LINK.symlink_to(DATA_ROOT, target_is_directory=True)
print("Resolved dataset:", DATA_ROOT)
print("UAD data link   :", DATA_LINK)


modality,IR,RGB,Thermal
split,,,
query,2135,2135,2135
test,31203,31203,31203
train,30711,30711,30711


Resolved dataset: /kaggle/input/datasets/cpkimhianh/whu-mars-s3clip
UAD data link   : /kaggle/working/pvuad_data/WHU-MARS


## 3. Pin source UAD + transplant **trainable TransReID backbone initialization**

1. kiểm tra đúng full MSMT17 TransReID checkpoint;
2. lấy các weight `base.*` tương thích với ViT backbone của UAD;
3. resize positional embedding **21×10 → 16×8** vì source TransReID dùng stride 12 còn target UAD mạnh cũ giữ stride 16;
4. **không** copy SIE camera, JPM local branches, BNNeck/classifier của MSMT17; riêng trained global branch `b1` được map vào block 11 + final norm của UAD;
5. tất cả weight đã transplant vẫn `requires_grad=True` và được E2/E3 update trên WHU-MARS.

Smoke test sẽ dừng notebook trước khi train nếu weight transplant sai, output không còn 768-D, hoặc gradient không đi vào attention/patch embedding.


In [4]:
PINNED_UAD_COMMIT = "3e2a07314119402586c76096e87d40420894ad30"
OFFICIAL_REPO_URL = "https://github.com/msm8976/WHU-MARS.git"
WORK_REPO = Path("/kaggle/working/WHU_MARS_official_pinned")
UAD_DIR = WORK_REPO / "CVPR26_UAD"

if WORK_REPO.exists():
    resolved = WORK_REPO.resolve()
    if not str(resolved).startswith("/kaggle/working/"):
        raise RuntimeError(f"Unsafe generated-source path: {resolved}")
    shutil.rmtree(WORK_REPO)

def attached_source_candidates(base="/kaggle/input"):
    candidates = []
    base = Path(base)
    if not base.exists():
        return candidates
    for root, dirs, _files in os.walk(base):
        root_path = Path(root)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
            continue
        if root_path.name != "CVPR26_UAD" or not (root_path / "train.py").is_file():
            continue
        repo_root = root_path.parent
        declared_commit = None
        manifest_path = root_path / "PVUAD_patch_manifest.json"
        if manifest_path.is_file():
            try:
                declared_commit = json.loads(
                    manifest_path.read_text(encoding="utf-8")
                ).get("base_commit")
            except Exception:
                declared_commit = None
        if (repo_root / ".git").is_dir():
            try:
                declared_commit = subprocess.check_output(
                    ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
                    text=True,
                ).strip()
            except Exception:
                pass
        if declared_commit != PINNED_UAD_COMMIT:
            # Never auto-select an unverified source tree. With no pinned
            # prior output, fall back to cloning the exact official commit.
            dirs[:] = []
            continue
        score = (
            2,
            1 if manifest_path.is_file() else 0,
            1 if "notebooks" in str(repo_root).lower() else 0,
        )
        candidates.append((score, repo_root, declared_commit))
        dirs[:] = []
    return sorted(candidates, key=lambda item: (item[0], str(item[1])), reverse=True)

source = Path(SOURCE_DIR_OVERRIDE) if SOURCE_DIR_OVERRIDE else None
if source is None:
    attached_sources = attached_source_candidates()
    if attached_sources:
        print("Attached UAD source candidates:")
        display(pd.DataFrame([
            {
                "path": str(path),
                "base_commit": commit,
                "selected": index == 0,
            }
            for index, (_score, path, commit) in enumerate(attached_sources)
        ]))
        source = attached_sources[0][1]

if source is not None:
    if (source / "CVPR26_UAD" / "train.py").is_file():
        shutil.copytree(source, WORK_REPO)
    elif source.name == "CVPR26_UAD" and (source / "train.py").is_file():
        WORK_REPO.mkdir(parents=True)
        shutil.copytree(source, UAD_DIR)
    else:
        raise FileNotFoundError(
            f"Attached/override source is not a WHU-MARS source tree: {source}"
        )
    print("Reusing attached UAD source:", source)
else:
    clone_command = [
        "git", "clone", "--no-checkout", "--filter=blob:none",
        OFFICIAL_REPO_URL, str(WORK_REPO),
    ]
    try:
        subprocess.run(clone_command, check=True)
        subprocess.run(
            ["git", "-C", str(WORK_REPO), "checkout", "--detach", PINNED_UAD_COMMIT],
            check=True,
        )
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "Cannot clone the pinned official UAD source. Enable Kaggle Internet or attach "
            "the previous notebook output/source and set SOURCE_DIR_OVERRIDE."
        ) from error

if not (UAD_DIR / "train.py").is_file():
    raise FileNotFoundError(f"Missing UAD train.py under {UAD_DIR}")
if (WORK_REPO / ".git").is_dir():
    head = subprocess.check_output(
        ["git", "-C", str(WORK_REPO), "rev-parse", "HEAD"], text=True
    ).strip()
    if head != PINNED_UAD_COMMIT:
        raise RuntimeError(f"Source commit mismatch: expected {PINNED_UAD_COMMIT}, got {head}")

OVERLAY_ARCHIVE_B64 = "..."
overlay_bytes = base64.b64decode(OVERLAY_ARCHIVE_B64)
with tarfile.open(fileobj=io.BytesIO(overlay_bytes), mode="r:gz") as archive:
    members = archive.getmembers()
    for member in members:
        destination = (UAD_DIR / member.name).resolve()
        if not str(destination).startswith(str(UAD_DIR.resolve()) + os.sep):
            raise RuntimeError(f"Unsafe overlay member: {member.name}")
    archive.extractall(UAD_DIR)

OVERLAY_HASHES = {'config/defaults.py': '839faddc24b14d21f073c0a2b588406cd24ac50651304c7d7dcdbe8d7416b89e', 'configs/PVUAD.yml': '1aa85aa03deee5b9449c71b4599fe630dcff6b11a454b51b458321131e23713a', 'datasets/bases.py': '7712bac25aee214f38a913cf3aad1ce4ef33b9a713d7e177ad67ec17932c016d', 'datasets/make_dataloader.py': '136cdc2da834f6ab181dc35c9c8f9a9cae085fec493685d730fde545cd4afe21', 'datasets/pvuad_sampler.py': 'c5c5e445a23e04d3a19bde3e49ea814d7c37ee8d540b70acd1254c19f3717220', 'datasets/pvuad_transforms.py': '056fc9f25aba6361d4612bcc313f0cc2d175e2b695757f6912e09fc7b5fc61c0', 'datasets/sampler_ddp.py': 'e4c455f7c7afbfac09a503a924c7b09586dbdb8eda6c88735ad1a3e37599bf7c', 'datasets/whu_mars.py': '11d3a0c501912a9887d78601df01afb6495d2f859835a6b69d2079135dba24b3', 'model/backbones/vit_pytorch.py': '723305c1a05e204b744ddb0ff7873b5e02840329e01787200173c9064ed3ad05', 'model/make_model.py': '8b7db7f5a190041beddf2788fc8820abe30f6f75b9b31818a402fef101b517ba', 'processor/processor.py': '9f678d53aa8fe6b7e01ba40242a1fb18d96e1a0ddf8f1046d5439124781b55df', 'train.py': '8da1552749d67272ca6d8078ac3dd8c1df4183924d7795a6e7b0355eecf18ff5'}
for relative, expected_hash in OVERLAY_HASHES.items():
    actual_hash = hashlib.sha256((UAD_DIR / relative).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"Overlay hash mismatch: {relative}")
patch_manifest = {
    "base_repo": OFFICIAL_REPO_URL,
    "base_commit": PINNED_UAD_COMMIT,
    "overlay_variant": "transreid_init_trainable_768_global_b1",
    "overlay_sha256": OVERLAY_HASHES,
}
(UAD_DIR / "PVUAD_patch_manifest.json").write_text(
    json.dumps(patch_manifest, indent=2), encoding="utf-8"
)
subprocess.run([sys.executable, "-m", "compileall", "-q", str(UAD_DIR)], check=True)
print("Pinned and patched source:", UAD_DIR)



TRANSREID_MODEL_NAME = "vit_transreid_msmt.pth"
TRANSREID_GDRIVE_ID = "1x6Na97ycxS0t2Dn_0iRKWe1U5ccIqASK"
TRANSREID_SOURCE_STRIDE = [12, 12]

def find_named_file_without_images(base, filename):
    base = Path(base)
    if not base.exists():
        return []
    found = []
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        if filename in files:
            found.append(root_path / filename)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
    return sorted(found, key=lambda item: str(item))


def torch_load_cpu(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def unwrap_checkpoint_state(payload):
    if isinstance(payload, dict):
        if "model" in payload and isinstance(payload["model"], dict):
            payload = payload["model"]
        elif "state_dict" in payload and isinstance(payload["state_dict"], dict):
            payload = payload["state_dict"]
    if not isinstance(payload, dict):
        raise ValueError("Checkpoint does not contain a state_dict")
    return {str(key).replace("module.", ""): value for key, value in payload.items()}


TRANSREID_CACHE = Path("/kaggle/working") / TRANSREID_MODEL_NAME
checkpoint_override = TRANSREID_CHECKPOINT_OVERRIDE or PRETRAIN_PATH_OVERRIDE
if checkpoint_override:
    source_checkpoint = Path(checkpoint_override)
    if not source_checkpoint.is_file():
        raise FileNotFoundError(source_checkpoint)
    if source_checkpoint.resolve() != TRANSREID_CACHE.resolve():
        shutil.copy2(source_checkpoint, TRANSREID_CACHE)
elif TRANSREID_CACHE.is_file():
    pass
else:
    attached = find_named_file_without_images("/kaggle/input", TRANSREID_MODEL_NAME)
    if attached:
        print("Caching attached pretrained TransReID checkpoint...")
        shutil.copy2(attached[0], TRANSREID_CACHE)
    else:
        print("Downloading official MSMT17 TransReID* ViT checkpoint with gdown...")
        try:
            import gdown
            result = gdown.download(
                id=TRANSREID_GDRIVE_ID,
                output=str(TRANSREID_CACHE),
                quiet=False,
            )
            if not result:
                raise RuntimeError("gdown returned no output path")
        except Exception as error:
            raise RuntimeError(
                "Cannot obtain vit_transreid_msmt.pth. Enable Kaggle Internet, "
                "or attach the official checkpoint as a Kaggle Dataset and set "
                "TRANSREID_CHECKPOINT_OVERRIDE."
            ) from error

if not TRANSREID_CACHE.is_file():
    raise FileNotFoundError(TRANSREID_CACHE)
if TRANSREID_CACHE.stat().st_size < 300_000_000:
    raise RuntimeError(
        "TransReID checkpoint is unexpectedly small; this is likely an HTML/"
        "partial download: {} bytes".format(TRANSREID_CACHE.stat().st_size)
    )

payload = torch_load_cpu(TRANSREID_CACHE)
tr_state = unwrap_checkpoint_state(payload)
required_transreid_keys = {
    "base.cls_token",
    "base.pos_embed",
    "base.patch_embed.proj.weight",
    "base.blocks.0.attn.qkv.weight",
    "base.blocks.11.attn.qkv.weight",
    "base.norm.weight",
    # These two are validated only to make sure this is the official full
    # TransReID checkpoint; they are intentionally NOT transplanted into UAD.
    "base.sie_embed",
    "b1.0.attn.qkv.weight",
    "b2.0.attn.qkv.weight",
}
missing = sorted(required_transreid_keys - set(tr_state))
if missing:
    raise RuntimeError(
        "Attached/downloaded file is not the expected full MSMT17 TransReID "
        f"checkpoint; missing keys: {missing}"
    )
if tr_state["base.cls_token"].shape[-1] != 768:
    raise RuntimeError("Expected ViT-B hidden dimension 768.")
if tuple(tr_state["base.sie_embed"].shape) != (15, 1, 768):
    raise RuntimeError(
        "Unexpected MSMT17 source SIE shape: "
        + str(tuple(tr_state["base.sie_embed"].shape))
    )
print(
    "Validated pretrained TransReID source:",
    {
        "path": str(TRANSREID_CACHE),
        "size_MiB": round(TRANSREID_CACHE.stat().st_size / 2**20, 1),
        "source_pos_embed": tuple(tr_state["base.pos_embed"].shape),
        "source_SIE": tuple(tr_state["base.sie_embed"].shape),
        "source_has_JPM": True,
        "target_embedding_dim": 768,
        "target_stride": TRANSREID_TARGET_STRIDE,
        "target_SIE": False,
        "target_JPM": False,
        "backbone_will_be_trainable": True,
    },
)
del payload, tr_state

TRANSREID_PATH = TRANSREID_CACHE
PRETRAIN_PATH = TRANSREID_PATH
print("Persistent TransReID initialization checkpoint:", PRETRAIN_PATH)


if RUN_TRANSREID_INIT_SMOKE_TEST:
    print("Running trainable 768-D TransReID-initialized UAD smoke test...")
    TRANSREID_INIT_AUDIT_PATH = Path(
        "/kaggle/working/transreid_trainable_init_audit.json"
    )
    smoke_script = r"""
import json
import sys
from pathlib import Path
import torch

uad_dir = Path(sys.argv[1])
checkpoint = Path(sys.argv[2])
audit_path = Path(sys.argv[3])
sys.path.insert(0, str(uad_dir))

from config import cfg
from model import make_model

def load_state(path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    if isinstance(payload, dict):
        if "model" in payload and isinstance(payload["model"], dict):
            payload = payload["model"]
        elif "state_dict" in payload and isinstance(payload["state_dict"], dict):
            payload = payload["state_dict"]
    return {str(k).replace("module.", ""): v for k, v in payload.items()}

cfg.merge_from_file(str(uad_dir / "configs/PVUAD.yml"))
cfg.MODEL.PRETRAIN_CHOICE = "transreid"
cfg.MODEL.PRETRAIN_PATH = str(checkpoint)
cfg.MODEL.STRIDE_SIZE = [16, 16]
cfg.MODEL.SIE_CAMERA = False
cfg.MODEL.SIE_VIEW = False
cfg.PVUAD.VFPROCA = True
cfg.PVUAD.LOCAL_CONSISTENCY = False
cfg.PVUAD.TIR_DISTILLATION = False

model = make_model(cfg, num_class=500, camera_num=8, view_num=0).cuda().train()
if model.transreid_init_audit is None:
    raise RuntimeError("Missing TransReID initialization audit")
if model.in_planes != 768:
    raise RuntimeError("UAD backbone must remain 768-D")
if hasattr(model.base, "sie_embed"):
    raise RuntimeError("Source-camera SIE must not be copied into target UAD")
if not all(parameter.requires_grad for parameter in model.base.parameters()):
    raise RuntimeError("TransReID-initialized backbone is unexpectedly frozen")

source = load_state(checkpoint)
source_qkv = source["base.blocks.0.attn.qkv.weight"]
target_qkv = model.base.blocks[0].attn.qkv.weight.detach().cpu()
init_qkv_max_abs = float((source_qkv - target_qkv).abs().max())
source_global_final_qkv = source["b1.0.attn.qkv.weight"]
target_final_qkv = model.base.blocks[11].attn.qkv.weight.detach().cpu()
global_final_qkv_max_abs = float(
    (source_global_final_qkv - target_final_qkv).abs().max()
)
if init_qkv_max_abs > 1e-7 or global_final_qkv_max_abs > 1e-7:
    raise RuntimeError(
        "TransReID weights were not transplanted exactly: "
        f"shared_qkv={init_qkv_max_abs}, global_final_qkv={global_final_qkv_max_abs}"
    )

images = [
    torch.randn(1, 3, 256, 128, device="cuda"),
    torch.randn(1, 3, 256, 128, device="cuda"),
    torch.randn(1, 3, 256, 128, device="cuda"),
]
output = model(x=images, mode=0)
cls_score, global_feat = output[0], output[1]
if tuple(global_feat.shape) != (3, 768):
    raise RuntimeError(f"Unexpected UAD feature shape: {tuple(global_feat.shape)}")
loss = cls_score.float().square().mean() + 0.01 * global_feat[:, :16].float().square().mean()
loss.backward()

qkv_grad = model.base.blocks[0].attn.qkv.weight.grad
patch_grad = model.base.patch_embed.proj.weight.grad
if qkv_grad is None or patch_grad is None:
    raise RuntimeError("No gradient reached the TransReID-initialized backbone")
qkv_grad_norm = float(qkv_grad.float().norm().detach().cpu())
patch_grad_norm = float(patch_grad.float().norm().detach().cpu())
if not (qkv_grad_norm > 0 and patch_grad_norm > 0):
    raise RuntimeError(
        f"Backbone gradients must be nonzero; qkv={qkv_grad_norm}, "
        f"patch={patch_grad_norm}"
    )

audit = dict(model.transreid_init_audit)
audit.update({
    "smoke_test": "OK",
    "feature_shape": list(global_feat.shape),
    "init_qkv_max_abs": init_qkv_max_abs,
    "global_final_qkv_max_abs": global_final_qkv_max_abs,
    "qkv_grad_norm": qkv_grad_norm,
    "patch_grad_norm": patch_grad_norm,
    "backbone_all_trainable": True,
    "source_sie_used": False,
    "source_jpm_local_branches_used": False,
    "source_jpm_global_final_branch_used": True,
    "target_stride": [16, 16],
})
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("TRANSREID_TRAINABLE_INIT_SMOKE_OK", audit)
"""
    smoke_env = os.environ.copy()
    smoke_env["CUDA_VISIBLE_DEVICES"] = "0"
    smoke_env["PYTHONPATH"] = str(UAD_DIR) + os.pathsep + smoke_env.get(
        "PYTHONPATH", ""
    )
    result = subprocess.run(
        [
            sys.executable,
            "-c",
            smoke_script,
            str(UAD_DIR),
            str(TRANSREID_PATH),
            str(TRANSREID_INIT_AUDIT_PATH),
        ],
        cwd=UAD_DIR,
        env=smoke_env,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "Trainable TransReID initialization smoke test failed. "
            "Do not start E2 until the initialization path is fixed."
        )


Attached UAD source candidates:


fatal: detected dubious ownership in repository at '/kaggle/input/notebooks/liennnnnn/pvuad-e2-e3-transreid-init-trainable-kaggle-2xt4/WHU_MARS_official_pinned'
To add an exception for this directory, call:

	git config --global --add safe.directory /kaggle/input/notebooks/liennnnnn/pvuad-e2-e3-transreid-init-trainable-kaggle-2xt4/WHU_MARS_official_pinned


,path,base_commit,selected
0,/kaggle/input/notebooks/liennnnnn/pvuad-e2-e3-...,3e2a07314119402586c76096e87d40420894ad30,True


Reusing attached UAD source: /kaggle/input/notebooks/liennnnnn/pvuad-e2-e3-transreid-init-trainable-kaggle-2xt4/WHU_MARS_official_pinned
Pinned and patched source: /kaggle/working/WHU_MARS_official_pinned/CVPR26_UAD


/tmp/ipykernel_23/786275482.py:116: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(UAD_DIR)


Caching attached pretrained TransReID checkpoint...
Validated pretrained TransReID source: {'path': '/kaggle/working/vit_transreid_msmt.pth', 'size_MiB': 399.8, 'source_pos_embed': (1, 211, 768), 'source_SIE': (15, 1, 768), 'source_has_JPM': True, 'target_embedding_dim': 768, 'target_stride': [16, 16], 'target_SIE': False, 'target_JPM': False, 'backbone_will_be_trainable': True}
Persistent TransReID initialization checkpoint: /kaggle/working/vit_transreid_msmt.pth
Running trainable 768-D TransReID-initialized UAD smoke test...
using Transformer_type: vit_base_in as a backbone
using stride: [16, 16], and patch number is num_y16 * num_x8
TransReID -> trainable UAD backbone initialization: {'checkpoint': '/kaggle/working/vit_transreid_msmt.pth', 'loaded_tensor_count': 152, 'target_tensor_count': 152, 'mismatch_count': 0, 'resized_pos_embed': {'source_shape': [1, 211, 768], 'target_shape': [1, 129, 768], 'target_grid': [16, 8]}, 'source_sie_loaded': False, 'source_jpm_local_branches_loaded

## 4. Pairing audit và state machine resume

Thứ tự ưu tiên vẫn là **E3 resume → E2 completed → E2 resume → E2 fresh**, nhưng chỉ checkpoint thuộc pipeline tag mới `e2_e3_transreid_init_trainable768_b24_v1` mới được auto-resume.

Điều này ngăn notebook vô tình lấy checkpoint của:
- UAD ImageNet-init cũ;
- frozen TransReID-JPM 3840-D;
- các experiment E2/E3 khác.


In [5]:
FILENAME_RE = re.compile(
    r"(?P<pid>\d+)_c(?P<camid>\d+)_m(?P<modid>\d+)_f(?P<frameid>\d+)",
    re.IGNORECASE,
)

PIPELINE_ROOT = Path("/kaggle/working/pvuad_runs") / PIPELINE_TAG
E2_RUN_DIR = PIPELINE_ROOT / "e2_final"
E3_RUN_DIR = PIPELINE_ROOT / "e3_tdh"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
E2_RUN_DIR.mkdir(parents=True, exist_ok=True)
E3_RUN_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_STATE_PATH = PIPELINE_ROOT / "pipeline_state.json"

INIT_AUDIT_SOURCE = Path("/kaggle/working/transreid_trainable_init_audit.json")
INIT_AUDIT_DEST = PIPELINE_ROOT / "transreid_trainable_init_audit.json"
if INIT_AUDIT_SOURCE.is_file() and not INIT_AUDIT_DEST.is_file():
    shutil.copy2(INIT_AUDIT_SOURCE, INIT_AUDIT_DEST)


def read_json(path):
    path = Path(path)
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as error:
        print("Ignoring unreadable JSON:", path, error)
        return {}


def write_pipeline_state(status, active_stage, **extra):
    payload = {
        "pipeline": PIPELINE_NAME,
        "pipeline_tag": PIPELINE_TAG,
        "status": status,
        "active_stage": active_stage,
        "updated_at_unix": _time.time(),
        "notebook_started_unix": NOTEBOOK_SESSION_STARTED_UNIX,
        "session_deadline_unix": SESSION_DEADLINE_UNIX,
        "remaining_minutes": max(
            0.0, (SESSION_DEADLINE_UNIX - _time.time()) / 60.0
        ),
        **extra,
    }
    temporary = PIPELINE_STATE_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(temporary, PIPELINE_STATE_PATH)
    return payload


def remaining_minutes():
    return max(0.0, (SESSION_DEADLINE_UNIX - _time.time()) / 60.0)


def copy_if_present(source_dir, destination_dir, filenames):
    if source_dir is None:
        return
    source_dir = Path(source_dir)
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        source_path = source_dir / filename
        destination_path = destination_dir / filename
        if source_path.is_file() and source_path.resolve() != destination_path.resolve():
            if not destination_path.exists():
                print("Preserving:", filename)
                shutil.copy2(source_path, destination_path)


def status_progress(status):
    return status.get("progress") or {}


def candidate_sort_key(candidate):
    status = candidate["status"]
    progress = status_progress(status)
    path_text = str(candidate["root"]).lower()
    return (
        int(PIPELINE_TAG.lower() in path_text),
        int(status.get("status") == "completed"),
        int(progress.get("epoch") or 0),
        int(bool(progress.get("epoch_complete", False))),
        int(progress.get("next_iteration") or 0),
        int(status.get("global_step") or 0),
        float(status.get("saved_at_unix") or candidate["checkpoint"].stat().st_mtime),
    )


def scan_stage_candidates(base, experiment):
    base = Path(base)
    if not base.exists():
        return []
    candidates = []
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
            continue
        if "pvuad_last_checkpoint.pth" not in files:
            continue
        checkpoint = root_path / "pvuad_last_checkpoint.pth"
        status = read_json(root_path / "session_status.json")
        summary = read_json(root_path / "training_summary.json")
        manifest = read_json(root_path / "run_manifest.json")
        declared = status.get("experiment", summary.get("experiment"))
        lowered = str(root_path).lower()
        legacy_match = (
            experiment == "E2" and "final_e2" in lowered
        ) or (
            experiment == "E3-TDH" and "e3_tdh" in lowered
        )
        if declared != experiment and not legacy_match:
            continue
        if status.get("stage", summary.get("stage")) not in (None, "final"):
            continue
        # Reject a manifest that clearly belongs to a different batch/LR
        # recipe. Missing legacy fields are tolerated, but contradictory
        # fields are not.
        recipe = manifest.get("recipe") or manifest.get("effective_training") or {}
        expected = (
            {"global_batch": E2_GLOBAL_BATCH, "epochs": E2_EPOCHS}
            if experiment == "E2"
            else {"global_batch": E3_GLOBAL_BATCH, "epochs": E3_EPOCHS}
        )
        incompatible = any(
            key in recipe and int(recipe[key]) != int(value)
            for key, value in expected.items()
        )
        lr_value = recipe.get("base_lr", recipe.get("backbone_lr"))
        expected_lr = E2_BASE_LR if experiment == "E2" else E3_BASE_LR
        if lr_value is not None and abs(float(lr_value) - expected_lr) > 1e-12:
            incompatible = True
        if incompatible:
            print("Skipping incompatible stage candidate:", root_path)
            continue
        candidates.append({
            "root": root_path,
            "checkpoint": checkpoint,
            "model": root_path / "pvuad_best_model.pth",
            "status": status,
            "summary": summary,
            "manifest": manifest,
        })
        dirs[:] = []
    return sorted(candidates, key=candidate_sort_key, reverse=True)


def candidate_table(candidates, label):
    if not candidates:
        return
    print(label)
    display(pd.DataFrame([{
        "root": str(item["root"]),
        "status": item["status"].get("status", "legacy_checkpoint"),
        "epoch": status_progress(item["status"]).get("epoch"),
        "next_iteration": status_progress(item["status"]).get("next_iteration"),
        "global_step": item["status"].get("global_step"),
        "best_mAP": item["status"].get("best_mAP_percent"),
        "selected": index == 0,
    } for index, item in enumerate(candidates)]))


# Pairing audit is invariant across sessions. Reuse a validated cached CSV when
# available; otherwise compute it once from the official train split.
PAIRING_AUDIT_PATH = PIPELINE_ROOT / "pairing_audit.csv"
cached_audits = []
if RUN_METADATA_PAIRING_CHECK and not PAIRING_AUDIT_PATH.is_file():
    for root, dirs, files in os.walk("/kaggle/input"):
        root_path = Path(root)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
            continue
        if "pairing_audit.csv" in files:
            cached_audits.append(root_path / "pairing_audit.csv")

pairing_frame = None
for cached in sorted(cached_audits, key=lambda item: str(item)):
    try:
        frame = pd.read_csv(cached)
        required = {"pid", "paired_frames", "ground", "aerial"}
        if len(frame) == 500 and required.issubset(frame.columns):
            if frame["paired_frames"].ge(1).all():
                pairing_frame = frame
                shutil.copy2(cached, PAIRING_AUDIT_PATH)
                print("Reused pairing audit:", cached)
                break
    except Exception:
        pass

if RUN_METADATA_PAIRING_CHECK and pairing_frame is None:
    keys_by_modality = {}
    for expected_modid, modality in enumerate(MODALITIES, start=1):
        by_pid = defaultdict(set)
        for path in sorted((DATA_ROOT / "train" / modality).glob("*.jpg")):
            match = FILENAME_RE.search(path.stem)
            if match is None:
                raise ValueError(f"Unexpected filename: {path.name}")
            pid = int(match.group("pid"))
            camid = int(match.group("camid"))
            modid = int(match.group("modid"))
            frameid = int(match.group("frameid"))
            if modid != expected_modid:
                raise ValueError(f"Folder/modality mismatch: {path}")
            by_pid[pid].add((camid, frameid))
        keys_by_modality[modality] = by_pid

    pids = sorted(set.intersection(*[set(mapping) for mapping in keys_by_modality.values()]))
    pairing_rows = []
    for pid in pids:
        common = set.intersection(*[
            keys_by_modality[modality][pid] for modality in MODALITIES
        ])
        ground = sum(camid <= 5 for camid, _frameid in common)
        aerial = sum(camid >= 6 for camid, _frameid in common)
        pairing_rows.append((pid, len(common), ground, aerial))
    pairing_frame = pd.DataFrame(
        pairing_rows, columns=["pid", "paired_frames", "ground", "aerial"]
    )
    pairing_frame["view_coverage"] = np.select(
        [
            pairing_frame["ground"].gt(0) & pairing_frame["aerial"].gt(0),
            pairing_frame["ground"].gt(0),
            pairing_frame["aerial"].gt(0),
        ],
        ["Ground+Aerial", "Ground-only", "Aerial-only"],
        default="No paired frame",
    )
    if len(pairing_frame) != 500:
        raise RuntimeError(f"Expected 500 train IDs, got {len(pairing_frame)}")
    no_pairs = pairing_frame.loc[
        pairing_frame["paired_frames"].eq(0), "pid"
    ].tolist()
    if no_pairs:
        raise RuntimeError(f"IDs without a complete RGB/NIR/TIR frame: {no_pairs[:20]}")
    pairing_frame.to_csv(PAIRING_AUDIT_PATH, index=False)

if pairing_frame is not None:
    if "view_coverage" not in pairing_frame.columns:
        pairing_frame["view_coverage"] = np.select(
            [
                pairing_frame["ground"].gt(0) & pairing_frame["aerial"].gt(0),
                pairing_frame["ground"].gt(0),
                pairing_frame["aerial"].gt(0),
            ],
            ["Ground+Aerial", "Ground-only", "Aerial-only"],
            default="No paired frame",
        )
    print("IDs with complete modality coverage:", len(pairing_frame))
    display(
        pairing_frame["view_coverage"]
        .value_counts(dropna=False)
        .rename_axis("view_coverage")
        .reset_index(name="identities")
    )
    # Ground-only/Aerial-only identities are valid WHU-MARS data. The sampler
    # falls back safely because STRICT_PAIRED_DATA remains False.


if E3_RESUME_PATH_OVERRIDE:
    E3_CANDIDATES = [{
        "root": Path(E3_RESUME_PATH_OVERRIDE).parent,
        "checkpoint": Path(E3_RESUME_PATH_OVERRIDE),
        "model": Path(E3_RESUME_PATH_OVERRIDE).parent / "pvuad_best_model.pth",
        "status": read_json(Path(E3_RESUME_PATH_OVERRIDE).parent / "session_status.json"),
        "summary": {},
        "manifest": read_json(Path(E3_RESUME_PATH_OVERRIDE).parent / "run_manifest.json"),
    }]
else:
    E3_CANDIDATES = scan_stage_candidates(E3_RUN_DIR, "E3-TDH")
    if AUTO_RESUME:
        external_e3 = scan_stage_candidates("/kaggle/input", "E3-TDH")
        external_e3 = [
            item for item in external_e3
            if PIPELINE_TAG.lower() in str(item["root"]).lower()
        ]
        E3_CANDIDATES += external_e3
        unique = {str(item["checkpoint"]): item for item in E3_CANDIDATES}
        E3_CANDIDATES = sorted(unique.values(), key=candidate_sort_key, reverse=True)

if E2_RESUME_PATH_OVERRIDE:
    E2_CANDIDATES = [{
        "root": Path(E2_RESUME_PATH_OVERRIDE).parent,
        "checkpoint": Path(E2_RESUME_PATH_OVERRIDE),
        "model": Path(E2_RESUME_PATH_OVERRIDE).parent / "pvuad_best_model.pth",
        "status": read_json(Path(E2_RESUME_PATH_OVERRIDE).parent / "session_status.json"),
        "summary": {},
        "manifest": read_json(Path(E2_RESUME_PATH_OVERRIDE).parent / "run_manifest.json"),
    }]
else:
    E2_CANDIDATES = scan_stage_candidates(E2_RUN_DIR, "E2")
    if AUTO_RESUME:
        external_e2 = scan_stage_candidates("/kaggle/input", "E2")
        if not ALLOW_LEGACY_E2_BOOTSTRAP:
            external_e2 = [
                item for item in external_e2
                if PIPELINE_TAG.lower() in str(item["root"]).lower()
            ]
        E2_CANDIDATES += external_e2
        unique = {str(item["checkpoint"]): item for item in E2_CANDIDATES}
        E2_CANDIDATES = sorted(unique.values(), key=candidate_sort_key, reverse=True)

if E2_COMPLETED_DIR_OVERRIDE:
    override_root = Path(E2_COMPLETED_DIR_OVERRIDE)
    E2_CANDIDATES.insert(0, {
        "root": override_root,
        "checkpoint": override_root / "pvuad_last_checkpoint.pth",
        "model": override_root / "pvuad_best_model.pth",
        "status": read_json(override_root / "session_status.json"),
        "summary": read_json(override_root / "training_summary.json"),
        "manifest": read_json(override_root / "run_manifest.json"),
    })

candidate_table(E3_CANDIDATES, "E3 resume candidates (highest priority):")
candidate_table(E2_CANDIDATES, "E2 candidates:")

SELECTED_E3 = E3_CANDIDATES[0] if E3_CANDIDATES else None
SELECTED_E2 = E2_CANDIDATES[0] if E2_CANDIDATES else None

# Preserve compact E2 evidence even when a completed legacy E2 is used only as
# the warm-start for E3. Large model/checkpoint files stay at their selected
# input path until E3 has created its self-contained teacher/resume artifacts.
if SELECTED_E2 is not None:
    copy_if_present(
        SELECTED_E2["root"], E2_RUN_DIR,
        [
            "metrics_history.csv", "session_status.json", "training_summary.json",
            "pairing_audit.csv", "ablation_summary.csv", "effective_command.txt",
        ],
    )

if SELECTED_E3 is not None:
    ACTIVE_STAGE = "E3"
elif SELECTED_E2 is not None and SELECTED_E2["status"].get("status") == "completed":
    ACTIVE_STAGE = "E3"
elif SELECTED_E2 is not None:
    ACTIVE_STAGE = "E2"
else:
    ACTIVE_STAGE = "E2"

print("Selected pipeline stage:", ACTIVE_STAGE)
write_pipeline_state(
    "ready", ACTIVE_STAGE,
    selected_e2=str(SELECTED_E2["root"]) if SELECTED_E2 else None,
    selected_e3=str(SELECTED_E3["root"]) if SELECTED_E3 else None,
)

(PIPELINE_ROOT / "RESUME_NEXT_SESSION.txt").write_text(
    "PV-UAD TransReID-init trainable E2 -> E3 multi-session pipeline\n\n"
    "1. A partial session ends normally with e2_paused or e3_paused.\n"
    "2. Start every run with Save Version -> Save & Run All; Kaggle then commits "
    "the output automatically after this notebook exits cleanly.\n"
    "3. In a new session, add the immediately preceding version output as Input.\n"
    "4. Use this exact notebook, keep Settings unchanged, and Run All.\n"
    "5. The state machine resumes E3 first, otherwise completed/partial E2.\n\n"
    f"Pipeline folder: pvuad_runs/{PIPELINE_TAG}\n",
    encoding="utf-8",
)
print("Pipeline output:", PIPELINE_ROOT)


Reused pairing audit: /kaggle/input/notebooks/liennnnnn/pvuad-e2-e3-transreid-init-trainable-kaggle-2xt4/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/pairing_audit.csv
IDs with complete modality coverage: 500


,view_coverage,identities
0,Ground-only,284
1,Ground+Aerial,216


E2 candidates:


,root,status,epoch,next_iteration,global_step,best_mAP,selected
0,/kaggle/input/notebooks/liennnnnn/pvuad-e2-e3-...,completed,60,0,91710,11.665469,True


Preserving: metrics_history.csv
Preserving: session_status.json
Preserving: training_summary.json
Preserving: effective_command.txt
Selected pipeline stage: E3
Pipeline output: /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1


## 5. Chạy E2 → E3-TDH → flip-TTA

**Fresh E2:** load pretrained TransReID vào UAD ViT 768-D rồi train toàn bộ backbone trên WHU-MARS.  
**E2 resume:** không load TransReID lại; resume chính xác optimizer/model/RNG từ checkpoint.  
**E3:** warm-start từ E2 tốt nhất và dùng lại full TIR local/global vì backbone vẫn là trainable ViT có patch tokens.  
**Final:** standard evaluation + horizontal-flip TTA, không re-ranking.

Mỗi phiên dùng chung guard 10.5 giờ và checkpoint định kỳ để chạy qua nhiều Kaggle sessions.


In [6]:
environment = os.environ.copy()
environment.update({
    "CUDA_VISIBLE_DEVICES": ",".join(map(str, range(NUM_GPUS))),
    "PYTHONPATH": str(UAD_DIR) + os.pathsep + environment.get("PYTHONPATH", ""),
    "PYTHONUNBUFFERED": "1",
    "OMP_NUM_THREADS": "2",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
})


def stage_status(run_dir):
    return read_json(Path(run_dir) / "session_status.json")


def display_stage_status(name, status):
    progress = status_progress(status)
    display(pd.DataFrame([{
        "stage": name,
        "status": status.get("status"),
        "reason": status.get("reason"),
        "epoch": progress.get("epoch"),
        "next_iteration": progress.get("next_iteration"),
        "epoch_complete": progress.get("epoch_complete"),
        "global_step": status.get("global_step"),
        "best_mAP": status.get("best_mAP_percent"),
        "remaining_minutes": round(remaining_minutes(), 1),
    }]))


def run_command_checked(command, run_dir, stage_name):
    command_text = shlex.join(command)
    (Path(run_dir) / "effective_command.txt").write_text(
        command_text + "\n", encoding="utf-8"
    )
    print("\nLaunching", stage_name)
    print(command_text)
    if not RUN_TRAINING:
        print("RUN_TRAINING=False — command prepared but not launched.")
        return {}
    result = subprocess.run(command, cwd=UAD_DIR, env=environment, check=False)
    if result.returncode != 0:
        write_pipeline_state(
            "failed", stage_name,
            returncode=result.returncode,
            run_dir=str(run_dir),
        )
        raise subprocess.CalledProcessError(result.returncode, command)
    required = [
        Path(run_dir) / "pvuad_last_checkpoint.pth",
        Path(run_dir) / "session_status.json",
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(f"{stage_name} ended without required outputs: {missing}")
    status = stage_status(run_dir)
    if status.get("status") not in {
        "paused_time_limit", "completed", "running_checkpoint"
    }:
        raise RuntimeError(f"Unexpected {stage_name} status: {status}")
    display_stage_status(stage_name, status)
    return status


def make_stage_manifest(run_dir, stage_name, recipe, resume_path=None, **extra):
    payload = {
        "pipeline": PIPELINE_NAME,
        "pipeline_tag": PIPELINE_TAG,
        "stage_name": stage_name,
        "recipe": recipe,
        "dataset_variant": DATASET_VARIANT,
        "dataset_root": str(DATA_ROOT),
        "dataset_counts": coverage_rows,
        "source_commit": PINNED_UAD_COMMIT,
        "overlay_sha256": OVERLAY_HASHES,
        "resume_path": str(resume_path) if resume_path else None,
        "python": sys.version,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpus": gpu_rows,
        "seed": RANDOM_SEED,
        "session_guard": {
            "notebook_started_unix": NOTEBOOK_SESSION_STARTED_UNIX,
            "session_deadline_unix": SESSION_DEADLINE_UNIX,
            "hard_stop_hours": SESSION_HARD_STOP_HOURS,
            "stop_reserve_minutes": SESSION_STOP_RESERVE_MINUTES,
            "checkpoint_every_minutes": CHECKPOINT_EVERY_MINUTES,
            "time_check_every_steps": TIME_CHECK_EVERY_STEPS,
            "min_eval_remaining_minutes": MIN_EVAL_REMAINING_MINUTES,
        },
        **extra,
    }
    manifest_path = Path(run_dir) / "run_manifest.json"
    manifest_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    shutil.copy2(
        UAD_DIR / "PVUAD_patch_manifest.json",
        Path(run_dir) / "PVUAD_patch_manifest.json",
    )
    return manifest_path


def e2_command(resume_path, strict_resume=True):
    # Fresh E2 receives pretrained TransReID backbone initialization exactly
    # once. Resume sessions reconstruct the model from the saved UAD checkpoint,
    # so they must NOT apply the source initialization again.
    pretrain_choice = "none" if resume_path else "transreid"
    recipe = {
        "experiment": "E2", "stage": "final", "epochs": E2_EPOCHS,
        "global_batch": E2_GLOBAL_BATCH, "base_lr": E2_BASE_LR,
        "paired_view_sampler": True, "vfproca": True,
        "local_consistency": False, "hard_gpd": False,
        "initialization": "MSMT17_TransReID_trainable_backbone",
        "pretrain_choice_this_session": pretrain_choice,
        "embedding_dim": 768,
        "backbone_stride": [16, 16],
        "source_sie_used": False,
        "source_jpm_local_branches_used": False,
        "source_jpm_global_final_branch_used": True,
    }
    manifest_path = make_stage_manifest(
        E2_RUN_DIR, "E2", recipe, resume_path=resume_path
    )
    opts = [
        "MODEL.DIST_TRAIN", "True",
        "MODEL.PRETRAIN_CHOICE", pretrain_choice,
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "SOLVER.SEED", str(RANDOM_SEED),
        "SOLVER.MAX_EPOCHS", str(E2_EPOCHS),
        "SOLVER.IMS_PER_BATCH", str(E2_GLOBAL_BATCH),
        "SOLVER.BASE_LR", str(E2_BASE_LR),
        "SOLVER.EVAL_PERIOD", "10",
        "SOLVER.CHECKPOINT_PERIOD", "5",
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "OUTPUT_DIR", str(E2_RUN_DIR),
        "PVUAD.EXPERIMENT", "E2",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.VFPROCA", "True",
        "MODEL.PCA_LOSS_WEIGHT", "0.01",
        "PVUAD.VIEW_ALIGNMENT_WEIGHT", "0.5",
        "PVUAD.INVARIANT_ID_WEIGHT", "0.5",
        "PVUAD.SCENARIO_CLS_WEIGHT", "0.1",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.LOCAL_CONSISTENCY", "False",
        "PVUAD.SYNC_PAIRED_AUG", "False",
        "PVUAD.HARD_GPD", "False",
        "PVUAD.FINETUNE", "False",
        "PVUAD.TIR_DISTILLATION", "False",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", "0.0",
        "PVUAD.SCENARIO_HARD_GPD", "False",
        "PVUAD.FINAL_FLIP_TTA", "False",
        "PVUAD.SAVE_LAST_EVERY", "5",
        "PVUAD.RUN_MANIFEST", str(manifest_path),
        "PVUAD.SESSION_DEADLINE_UNIX", str(float(SESSION_DEADLINE_UNIX)),
        "PVUAD.STOP_RESERVE_MINUTES", str(float(SESSION_STOP_RESERVE_MINUTES)),
        "PVUAD.CHECKPOINT_EVERY_MINUTES", str(float(CHECKPOINT_EVERY_MINUTES)),
        "PVUAD.TIME_CHECK_EVERY_STEPS", str(int(TIME_CHECK_EVERY_STEPS)),
        "PVUAD.MIN_EVAL_REMAINING_MINUTES", str(float(MIN_EVAL_REMAINING_MINUTES)),
        "PVUAD.STRICT_RESUME_CONFIG", str(bool(strict_resume)),
    ]
    if resume_path:
        opts += ["PVUAD.RESUME_PATH", str(resume_path)]
    return [
        sys.executable, "-m", "torch.distributed.run", "--standalone",
        f"--nproc_per_node={NUM_GPUS}", "train.py", "--config_file",
        "configs/PVUAD.yml", *opts,
    ]


E2_STATUS = SELECTED_E2["status"] if SELECTED_E2 else {}
E2_RESUME_PATH = None
E2_MODEL_PATH = None
E2_CHECKPOINT_PATH = None
E2_STRICT_RESUME = True
E2_COMPLETED_THIS_SESSION = False

if ACTIVE_STAGE == "E2":
    if SELECTED_E2 is not None:
        E2_RESUME_PATH = SELECTED_E2["checkpoint"]
        selected_hashes = (
            SELECTED_E2.get("manifest", {}).get("overlay_sha256") or {}
        )
        selected_processor_hash = selected_hashes.get("processor/processor.py")
        current_processor_hash = OVERLAY_HASHES.get("processor/processor.py")
        # E2 v2.3 checkpoints predate E3's gated fields in the resume
        # signature. Their known E2 structure is compatible; disable only the
        # signature-key comparison while retaining strict state_dict/optimizer
        # structural loading. New combined checkpoints remain fully strict.
        if (
            selected_processor_hash
            and selected_processor_hash != current_processor_hash
        ):
            E2_STRICT_RESUME = False
            print(
                "Known legacy E2 overlay detected; structural resume checks "
                "remain enabled, signature-only strictness is relaxed."
            )
        copy_if_present(
            SELECTED_E2["root"], E2_RUN_DIR,
            [
                "metrics_history.csv", "pvuad_best_model.pth",
                "pairing_audit.csv", "ablation_summary.csv",
                "training_summary.json",
            ],
        )
    else:
        print("No checkpoint found: starting E2 from pretrained TransReID ViT weights; backbone is TRAINABLE.")

    write_pipeline_state(
        "e2_running", "E2",
        resume_path=str(E2_RESUME_PATH) if E2_RESUME_PATH else None,
    )
    E2_STATUS = run_command_checked(
        e2_command(E2_RESUME_PATH, E2_STRICT_RESUME), E2_RUN_DIR, "E2"
    )
    if not RUN_TRAINING:
        write_pipeline_state("dry_run", "E2")
    elif E2_STATUS.get("status") in {"paused_time_limit", "running_checkpoint"}:
        write_pipeline_state(
            "e2_paused", "E2", e2_status=E2_STATUS,
            next_action="Attach this output to a new session and Run All.",
        )
        print("E2 PAUSED SAFELY. Save Version and continue in a new session.")
    elif E2_STATUS.get("status") == "completed":
        E2_MODEL_PATH = E2_RUN_DIR / "pvuad_best_model.pth"
        E2_CHECKPOINT_PATH = E2_RUN_DIR / "pvuad_last_checkpoint.pth"
        required = [E2_MODEL_PATH, E2_CHECKPOINT_PATH, E2_RUN_DIR / "metrics_history.csv"]
        missing = [str(path) for path in required if not path.is_file()]
        if missing:
            raise RuntimeError(f"Completed E2 is missing: {missing}")
        E2_COMPLETED_THIS_SESSION = True
        ACTIVE_STAGE = "E3"
else:
    if SELECTED_E2 is not None:
        E2_MODEL_PATH = SELECTED_E2["model"]
        E2_CHECKPOINT_PATH = SELECTED_E2["checkpoint"]


def create_teacher_cache(source_model, destination):
    source_model = Path(source_model)
    destination = Path(destination)
    if destination.is_file():
        return destination
    if not source_model.is_file():
        raise FileNotFoundError(source_model)
    try:
        payload = torch.load(source_model, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(source_model, map_location="cpu")
    if isinstance(payload, dict) and "model" in payload:
        payload = payload["model"]
    elif isinstance(payload, dict) and "state_dict" in payload:
        payload = payload["state_dict"]
    if not isinstance(payload, dict):
        raise ValueError("E2 model file does not contain a state dictionary")
    teacher_state = {}
    for key, value in payload.items():
        clean = str(key).replace("module.", "")
        teacher_state[clean] = value.half() if torch.is_floating_point(value) else value
    temporary = destination.with_suffix(".pth.tmp")
    torch.save(teacher_state, temporary)
    os.replace(temporary, destination)
    del payload, teacher_state
    return destination


E3_STATUS = SELECTED_E3["status"] if SELECTED_E3 else {}
E3_RESUME_PATH = SELECTED_E3["checkpoint"] if SELECTED_E3 else None
TEACHER_CACHE = E3_RUN_DIR / "e2_teacher_model_fp16.pth"

if SELECTED_E3 is not None:
    copy_if_present(
        SELECTED_E3["root"], E3_RUN_DIR,
        [
            "metrics_history.csv", "pvuad_best_model.pth", "pvuad_last_model.pth",
            "e2_teacher_model_fp16.pth", "tta_metrics.csv",
            "training_summary.json", "TRAINING_COMPLETE_BEFORE_TTA.json",
            "FINAL_TTA_FAILED.txt", "effective_test_command.txt",
        ],
    )


def e3_command(resume_path, student_model, prototype_checkpoint, teacher_cache):
    recipe = {
        "experiment": "E3-TDH", "stage": "final", "epochs": E3_EPOCHS,
        "global_batch": E3_GLOBAL_BATCH, "backbone_lr": E3_BASE_LR,
        "head_lr_multiplier": E3_HEAD_LR_MULTIPLIER,
        "tir_global_weight": TIR_GLOBAL_WEIGHT,
        "tir_local_weight": TIR_LOCAL_WEIGHT,
        "feature_preserve_weight": FEATURE_PRESERVE_WEIGHT,
        "scenario_hard_weight": SCENARIO_HARD_WEIGHT,
        "initialization_origin": "MSMT17_TransReID_then_WHUMARS_E2",
        "embedding_dim": 768,
        "backbone_stride": [16, 16],
    }
    manifest_path = make_stage_manifest(
        E3_RUN_DIR, "E3-TDH", recipe, resume_path=resume_path,
        e2_model_path=str(E2_MODEL_PATH) if E2_MODEL_PATH else None,
        e2_checkpoint_path=str(E2_CHECKPOINT_PATH) if E2_CHECKPOINT_PATH else None,
        teacher_model_path=str(teacher_cache),
    )
    opts = [
        "MODEL.DIST_TRAIN", "True",
        "MODEL.PRETRAIN_CHOICE", "none",
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "SOLVER.SEED", str(RANDOM_SEED),
        "SOLVER.MAX_EPOCHS", str(E3_EPOCHS),
        "SOLVER.IMS_PER_BATCH", str(E3_GLOBAL_BATCH),
        "SOLVER.BASE_LR", str(E3_BASE_LR),
        "SOLVER.WARMUP_EPOCHS", "1",
        "SOLVER.EVAL_PERIOD", "5",
        "SOLVER.CHECKPOINT_PERIOD", "5",
        "INPUT.RE_PROB", "0.0",
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "OUTPUT_DIR", str(E3_RUN_DIR),
        "PVUAD.EXPERIMENT", "E3-TDH",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.SYNC_PAIRED_AUG", "True",
        "PVUAD.VFPROCA", "True",
        "MODEL.PCA_LOSS_WEIGHT", "0.01",
        "PVUAD.VIEW_ALIGNMENT_WEIGHT", "0.5",
        "PVUAD.INVARIANT_ID_WEIGHT", "0.5",
        "PVUAD.SCENARIO_CLS_WEIGHT", "0.1",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.LOCAL_CONSISTENCY", "False",
        "PVUAD.HARD_GPD", "False",
        "PVUAD.FINETUNE", "True",
        "PVUAD.WARMSTART_MODEL_PATH", str(student_model),
        "PVUAD.WARMSTART_PROTOTYPE_PATH", (
            str(prototype_checkpoint) if prototype_checkpoint else ""
        ),
        "PVUAD.TEACHER_MODEL_PATH", str(teacher_cache),
        "PVUAD.HEAD_LR_MULTIPLIER", str(float(E3_HEAD_LR_MULTIPLIER)),
        "PVUAD.TIR_DISTILLATION", "True",
        "PVUAD.TIR_GLOBAL_WEIGHT", str(float(TIR_GLOBAL_WEIGHT)),
        "PVUAD.TIR_LOCAL_WEIGHT", str(float(TIR_LOCAL_WEIGHT)),
        "PVUAD.TIR_LOCAL_KEEP_RATIO", str(float(TIR_LOCAL_KEEP_RATIO)),
        "PVUAD.FEATURE_PRESERVE_WEIGHT", str(float(FEATURE_PRESERVE_WEIGHT)),
        "PVUAD.SCENARIO_HARD_GPD", "True",
        "PVUAD.SCENARIO_HARD_WEIGHT", str(float(SCENARIO_HARD_WEIGHT)),
        "PVUAD.SCENARIO_HARD_TOPK", str(int(SCENARIO_HARD_TOPK)),
        "PVUAD.SCENARIO_HARD_MARGIN", str(float(SCENARIO_HARD_MARGIN)),
        "PVUAD.SCENARIO_HARD_TAU", str(float(SCENARIO_HARD_TAU)),
        "PVUAD.SCENARIO_HARD_START_EPOCH", str(int(SCENARIO_HARD_START_EPOCH)),
        "PVUAD.SCENARIO_BANK_MOMENTUM", "0.2",
        "PVUAD.FINAL_FLIP_TTA", "False",
        "PVUAD.SAVE_LAST_EVERY", "5",
        "PVUAD.RUN_MANIFEST", str(manifest_path),
        "PVUAD.SESSION_DEADLINE_UNIX", str(float(SESSION_DEADLINE_UNIX)),
        "PVUAD.STOP_RESERVE_MINUTES", str(float(SESSION_STOP_RESERVE_MINUTES)),
        "PVUAD.CHECKPOINT_EVERY_MINUTES", str(float(CHECKPOINT_EVERY_MINUTES)),
        "PVUAD.TIME_CHECK_EVERY_STEPS", str(int(TIME_CHECK_EVERY_STEPS)),
        "PVUAD.MIN_EVAL_REMAINING_MINUTES", str(float(MIN_EVAL_REMAINING_MINUTES)),
        "PVUAD.STRICT_RESUME_CONFIG", "True",
    ]
    if resume_path:
        opts += ["PVUAD.RESUME_PATH", str(resume_path)]
    return [
        sys.executable, "-m", "torch.distributed.run", "--standalone",
        f"--nproc_per_node={NUM_GPUS}", "train.py", "--config_file",
        "configs/PVUAD.yml", *opts,
    ]


can_enter_e3 = (
    ACTIVE_STAGE == "E3"
    and (E3_STATUS.get("status") != "completed")
    and RUN_TRAINING
)
if (
    can_enter_e3
    and E2_COMPLETED_THIS_SESSION
    and ALWAYS_START_E3_IN_NEW_SESSION
):
    write_pipeline_state(
        "e2_completed_e3_pending", "E3",
        e2_status=E2_STATUS,
        remaining_minutes=remaining_minutes(),
        next_action=(
            "Kaggle will save this completed E2 output. Attach it to a new "
            "session and use Save & Run All; E3 will start automatically."
        ),
    )
    print(
        "E2 COMPLETE. This Kaggle version now ends cleanly by design. "
        "Attach its output to the next session; E3 will start there."
    )
    can_enter_e3 = False
elif can_enter_e3 and remaining_minutes() < MIN_E3_START_REMAINING_MINUTES:
    pending_status = "e3_paused" if E3_RESUME_PATH else "e2_completed_e3_pending"
    write_pipeline_state(
        pending_status, "E3",
        e2_status=E2_STATUS,
        remaining_minutes=remaining_minutes(),
        next_action="Save Version; attach this output and Run All in a new session.",
    )
    print(
        "Only {:.1f} minutes remain. E3 will start/resume in the next "
        "session.".format(remaining_minutes())
    )
    can_enter_e3 = False

if can_enter_e3:
    if E3_RESUME_PATH is not None:
        teacher_source = (
            SELECTED_E3["root"] / "e2_teacher_model_fp16.pth"
        )
        if not teacher_source.is_file() and E2_MODEL_PATH is not None:
            teacher_source = E2_MODEL_PATH
        if not TEACHER_CACHE.is_file():
            if teacher_source.name == "e2_teacher_model_fp16.pth":
                shutil.copy2(teacher_source, TEACHER_CACHE)
            else:
                create_teacher_cache(teacher_source, TEACHER_CACHE)
        student_model = TEACHER_CACHE
        prototype_checkpoint = None
    else:
        if E2_MODEL_PATH is None or not Path(E2_MODEL_PATH).is_file():
            raise FileNotFoundError(
                "E3 needs a completed E2 pvuad_best_model.pth. Attach the "
                "previous combined/E2 output or use E2_COMPLETED_DIR_OVERRIDE."
            )
        if E2_CHECKPOINT_PATH is None or not Path(E2_CHECKPOINT_PATH).is_file():
            raise FileNotFoundError(
                "E3 needs E2 pvuad_last_checkpoint.pth for prototype warm-start."
            )
        create_teacher_cache(E2_MODEL_PATH, TEACHER_CACHE)
        student_model = E2_MODEL_PATH
        prototype_checkpoint = E2_CHECKPOINT_PATH

    print("Frozen E2 teacher:", TEACHER_CACHE)
    write_pipeline_state(
        "e3_running", "E3",
        e2_status=E2_STATUS,
        e3_resume_path=str(E3_RESUME_PATH) if E3_RESUME_PATH else None,
    )
    E3_STATUS = run_command_checked(
        e3_command(
            E3_RESUME_PATH, student_model, prototype_checkpoint, TEACHER_CACHE
        ),
        E3_RUN_DIR,
        "E3-TDH",
    )
    if E3_STATUS.get("status") in {"paused_time_limit", "running_checkpoint"}:
        write_pipeline_state(
            "e3_paused", "E3", e2_status=E2_STATUS, e3_status=E3_STATUS,
            next_action="Attach this output to a new session and Run All.",
        )
        print("E3 PAUSED SAFELY. Save Version and continue in a new session.")


def run_final_tta(best_model):
    test_opts = [
        "MODEL.DIST_TRAIN", "False",
        "MODEL.PRETRAIN_CHOICE", "none",
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "TEST.WEIGHT", str(best_model),
        "OUTPUT_DIR", str(E3_RUN_DIR),
        "PVUAD.EXPERIMENT", "E3-TDH",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.VFPROCA", "True",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.TIR_DISTILLATION", "True",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", "0.0",
        "PVUAD.SCENARIO_HARD_GPD", "True",
        "PVUAD.FINAL_FLIP_TTA", "True",
    ]
    test_command = [
        sys.executable, "test.py", "--config_file", "configs/PVUAD.yml",
        *test_opts,
    ]
    (E3_RUN_DIR / "effective_test_command.txt").write_text(
        shlex.join(test_command) + "\n", encoding="utf-8"
    )
    (E3_RUN_DIR / "TRAINING_COMPLETE_BEFORE_TTA.json").write_text(
        json.dumps({
            "status": "training_complete",
            "best_model": str(best_model),
            "last_checkpoint": str(E3_RUN_DIR / "pvuad_last_checkpoint.pth"),
            "tta_pending": True,
        }, indent=2),
        encoding="utf-8",
    )
    print("Running final standard + horizontal-flip TTA evaluation...")
    result = subprocess.run(
        test_command, cwd=UAD_DIR, env=environment, check=False
    )
    tta_metrics_path = E3_RUN_DIR / "tta_metrics.csv"
    if result.returncode != 0 or not tta_metrics_path.is_file():
        warning = (
            "Final TTA failed after E3 training completed; checkpoints remain valid. "
            f"returncode={result.returncode}. Attach this output to retry TTA only."
        )
        (E3_RUN_DIR / "FINAL_TTA_FAILED.txt").write_text(
            warning + "\n", encoding="utf-8"
        )
        print("WARNING:", warning)
        return False
    (E3_RUN_DIR / "TRAINING_COMPLETE_BEFORE_TTA.json").write_text(
        json.dumps({
            "status": "training_and_tta_complete",
            "best_model": str(best_model),
            "tta_pending": False,
        }, indent=2),
        encoding="utf-8",
    )
    failed_marker = E3_RUN_DIR / "FINAL_TTA_FAILED.txt"
    if failed_marker.is_file():
        failed_marker.unlink()
    return True


E3_COMPLETED = E3_STATUS.get("status") == "completed"
if E3_COMPLETED:
    # When completion came from an attached output, preserve its full checkpoint
    # so the newly saved combined output remains independently resumable.
    if SELECTED_E3 is not None:
        copy_if_present(
            SELECTED_E3["root"], E3_RUN_DIR,
            ["pvuad_last_checkpoint.pth", "session_status.json"],
        )
    best_model = E3_RUN_DIR / "pvuad_best_model.pth"
    if not best_model.is_file() and SELECTED_E3 is not None:
        copy_if_present(
            SELECTED_E3["root"], E3_RUN_DIR, ["pvuad_best_model.pth"]
        )
    if not best_model.is_file():
        raise FileNotFoundError("Completed E3 is missing pvuad_best_model.pth")

    tta_path = E3_RUN_DIR / "tta_metrics.csv"
    if not tta_path.is_file() and RUN_FINAL_FLIP_TTA:
        if remaining_minutes() >= MIN_TTA_START_REMAINING_MINUTES:
            tta_ok = run_final_tta(best_model)
        else:
            tta_ok = False
            print(
                "E3 training is complete, but only {:.1f} minutes remain; "
                "TTA will run next session.".format(remaining_minutes())
            )
    else:
        tta_ok = tta_path.is_file()

    final_status = "completed" if (tta_ok or not RUN_FINAL_FLIP_TTA) else "e3_complete_tta_pending"
    write_pipeline_state(
        final_status, "DONE" if final_status == "completed" else "TTA",
        e2_status=E2_STATUS,
        e3_status=E3_STATUS,
        tta_complete=bool(tta_ok),
        next_action=(
            None if final_status == "completed"
            else "Attach this output and Run All; training will be skipped and TTA retried."
        ),
    )
elif ACTIVE_STAGE == "E3" and not can_enter_e3 and not RUN_TRAINING:
    write_pipeline_state("dry_run", "E3")


Frozen E2 teacher: /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh/e2_teacher_model_fp16.pth

Launching E3-TDH
/usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node=2 train.py --config_file configs/PVUAD.yml MODEL.DIST_TRAIN True MODEL.PRETRAIN_CHOICE none MODEL.PRETRAIN_PATH /kaggle/working/vit_transreid_msmt.pth MODEL.STRIDE_SIZE '[16, 16]' MODEL.SIE_CAMERA False MODEL.SIE_VIEW False DATASETS.ROOT_DIR /kaggle/working/pvuad_data DATALOADER.NUM_WORKERS 4 SOLVER.SEED 1234 SOLVER.MAX_EPOCHS 25 SOLVER.IMS_PER_BATCH 24 SOLVER.BASE_LR 0.0001 SOLVER.WARMUP_EPOCHS 1 SOLVER.EVAL_PERIOD 5 SOLVER.CHECKPOINT_PERIOD 5 INPUT.RE_PROB 0.0 TEST.IMS_PER_BATCH 256 TEST.RE_RANKING False TEST.TOP_K_EVAL 0 OUTPUT_DIR /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh PVUAD.EXPERIMENT E3-TDH PVUAD.STAGE final PVUAD.DEV_MODE False PVUAD.PAIRED_VIEW_SAMPLER True PVUAD.STRICT_PAIRED_DATA False PVUAD.SYNC_PAIRED_AUG True PVUAD.VFPROCA True MOD

[W909 06:47:57.340452833 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


2026-09-09 06:48:11,845 transreid INFO: Saving model in the path :/kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh
2026-09-09 06:48:11,845 transreid INFO: Namespace(config_file='configs/PVUAD.yml', opts=['MODEL.DIST_TRAIN', 'True', 'MODEL.PRETRAIN_CHOICE', 'none', 'MODEL.PRETRAIN_PATH', '/kaggle/working/vit_transreid_msmt.pth', 'MODEL.STRIDE_SIZE', '[16, 16]', 'MODEL.SIE_CAMERA', 'False', 'MODEL.SIE_VIEW', 'False', 'DATASETS.ROOT_DIR', '/kaggle/working/pvuad_data', 'DATALOADER.NUM_WORKERS', '4', 'SOLVER.SEED', '1234', 'SOLVER.MAX_EPOCHS', '25', 'SOLVER.IMS_PER_BATCH', '24', 'SOLVER.BASE_LR', '0.0001', 'SOLVER.WARMUP_EPOCHS', '1', 'SOLVER.EVAL_PERIOD', '5', 'SOLVER.CHECKPOINT_PERIOD', '5', 'INPUT.RE_PROB', '0.0', 'TEST.IMS_PER_BATCH', '256', 'TEST.RE_RANKING', 'False', 'TEST.TOP_K_EVAL', '0', 'OUTPUT_DIR', '/kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh', 'PVUAD.EXPERIMENT', 'E3-TDH', 'PVUAD.STAGE', 'final', 'PVUAD.DEV_MODE', 'False'

[W909 06:48:11.650158100 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W909 06:48:11.656058056 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=> WHU-MARS loaded
=> WHU-MARS loaded
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   500 |    92133 |         7
  query    |   500 |     6405 |         7
  gallery  |   500 |    93609 |         7
  ----------------------------------------
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   500 |    92133 |         7
  query    |   500 |     6405 |         7
  gallery  |   500 |    93609 |         7
  ----------------------------------------
DIST_TRAIN START
DIST_TRAIN START
PairedViewPKM: 500 IDs (216 dual-view, 284 single-view), 37,148 paired sample slots/epoch, 3 PID x 4 instances per GPU
using Transformer_type: vit_base_in as a backbone
using stride: [16, 16], and patch number is num_y16 * num_x8
PairedViewPKM: 500 IDs (216 dual-view, 284 single-view), 37,148 

/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


2026-09-09 07:18:25,681 transreid.train INFO: Epoch 3 done in 608.6s; loss 2.1765; acc 0.7806
2026-09-09 07:18:48,057 transreid.train INFO: Epoch[4] Iteration[50/1524] Loss 2.994 Acc 0.678 ID 1.559 TRI 0.138 Align 0.260 GPD 0.171 TIR-G 0.457 TIR-L 0.332 Keep 0.056 S-Hard 0.024 LR 9.38e-05
2026-09-09 07:18:48,057 transreid.train INFO: Epoch[4] Iteration[50/1524] Loss 3.305 Acc 0.643 ID 1.722 TRI 0.160 Align 0.266 GPD 0.189 TIR-G 0.462 TIR-L 0.338 Keep 0.057 S-Hard 0.024 LR 9.38e-05
2026-09-09 07:19:08,089 transreid.train INFO: Epoch[4] Iteration[100/1524] Loss 3.159 Acc 0.658 ID 1.643 TRI 0.147 Align 0.272 GPD 0.180 TIR-G 0.465 TIR-L 0.337 Keep 0.055 S-Hard 0.025 LR 9.38e-05
2026-09-09 07:19:08,089 transreid.train INFO: Epoch[4] Iteration[100/1524] Loss 3.043 Acc 0.670 ID 1.585 TRI 0.140 Align 0.267 GPD 0.167 TIR-G 0.456 TIR-L 0.331 Keep 0.055 S-Hard 0.024 LR 9.38e-05
2026-09-09 07:19:27,648 transreid.train INFO: Epoch[4] Iteration[150/1524] Loss 3.092 Acc 0.672 ID 1.610 TRI 0.138 Align

,stage,status,reason,epoch,next_iteration,epoch_complete,global_step,best_mAP,remaining_minutes
0,E3-TDH,completed,training_completed,25,0,True,38207,11.805238,306.6


Running final standard + horizontal-flip TTA evaluation...
2026-09-09 11:57:47,478 transreid INFO: Namespace(config_file='configs/PVUAD.yml', opts=['MODEL.DIST_TRAIN', 'False', 'MODEL.PRETRAIN_CHOICE', 'none', 'MODEL.PRETRAIN_PATH', '/kaggle/working/vit_transreid_msmt.pth', 'MODEL.STRIDE_SIZE', '[16, 16]', 'MODEL.SIE_CAMERA', 'False', 'MODEL.SIE_VIEW', 'False', 'DATASETS.ROOT_DIR', '/kaggle/working/pvuad_data', 'DATALOADER.NUM_WORKERS', '4', 'TEST.IMS_PER_BATCH', '256', 'TEST.RE_RANKING', 'False', 'TEST.TOP_K_EVAL', '0', 'TEST.WEIGHT', '/kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh/pvuad_best_model.pth', 'OUTPUT_DIR', '/kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh', 'PVUAD.EXPERIMENT', 'E3-TDH', 'PVUAD.STAGE', 'final', 'PVUAD.DEV_MODE', 'False', 'PVUAD.PAIRED_VIEW_SAMPLER', 'True', 'PVUAD.STRICT_PAIRED_DATA', 'False', 'PVUAD.VFPROCA', 'True', 'PVUAD.TEST_INVARIANT_BLEND', '0.5', 'PVUAD.TIR_DISTILLATION', 'True', 'PVUAD.FEATURE_P

## 6. Tổng hợp và đóng gói

`pvuad_e2_e3_transreid_init_trainable_reports.zip` chứa log/CSV/JSON nhẹ, bao gồm cả `transreid_trainable_init_audit.json` để xác nhận initialization/gradient.

Checkpoint model vẫn nằm riêng trong `pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/...`.


In [7]:
def best_metrics(run_dir, label):
    path = Path(run_dir) / "metrics_history.csv"
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    required = {"experiment", "stage", "epoch", "mAP", "Rank-1", "Rank-5", "Rank-10"}
    if frame.empty or not required.issubset(frame.columns):
        return None
    frame = frame.drop_duplicates(
        subset=["experiment", "stage", "epoch"], keep="last"
    )
    frame.to_csv(path, index=False)
    row = frame.loc[frame["mAP"].idxmax()]
    return {
        "method": label,
        "epoch": int(row["epoch"]),
        "mAP": float(row["mAP"]),
        "Rank-1": float(row["Rank-1"]),
        "Rank-5": float(row["Rank-5"]),
        "Rank-10": float(row["Rank-10"]),
    }


state = read_json(PIPELINE_STATE_PATH)
print("Pipeline status:", state.get("status"))
print("Active stage   :", state.get("active_stage"))
print("Next action    :", state.get("next_action"))
init_audit_path = PIPELINE_ROOT / "transreid_trainable_init_audit.json"
if init_audit_path.is_file():
    print("TransReID initialization audit:")
    display(pd.DataFrame([read_json(init_audit_path)]))

comparison = [{
    "method": "UAD paper",
    "epoch": np.nan,
    "mAP": 11.000,
    "Rank-1": 29.500,
    "Rank-5": 46.300,
    "Rank-10": 54.800,
}]
e2_best = best_metrics(E2_RUN_DIR, "E2 TransReID-init")
e3_best = best_metrics(E3_RUN_DIR, "E3-TDH TransReID-init")
if e2_best:
    comparison.append(e2_best)
if e3_best:
    comparison.append(e3_best)

tta_path = E3_RUN_DIR / "tta_metrics.csv"
if tta_path.is_file():
    tta = pd.read_csv(tta_path)
    if not tta.empty:
        row = tta.iloc[-1]
        comparison.append({
            "method": "E3-TDH TransReID-init flip TTA",
            "epoch": E3_EPOCHS,
            "mAP": float(row["mAP"]),
            "Rank-1": float(row["Rank-1"]),
            "Rank-5": float(row["Rank-5"]),
            "Rank-10": float(row["Rank-10"]),
        })

comparison_frame = pd.DataFrame(comparison)
comparison_frame.to_csv(PIPELINE_ROOT / "combined_results.csv", index=False)
display(comparison_frame.round(3))

REPORT_ZIP = Path("/kaggle/working") / "pvuad_e2_e3_transreid_init_trainable_reports.zip"
if REPORT_ZIP.exists():
    REPORT_ZIP.unlink()
report_suffixes = {".csv", ".json", ".txt", ".log", ".yml", ".yaml"}
with zipfile.ZipFile(REPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PIPELINE_ROOT.rglob("*")):
        if path.is_file() and path.suffix.lower() in report_suffixes:
            archive.write(path, arcname=str(path.relative_to(PIPELINE_ROOT)))

print("\nReport ZIP:", REPORT_ZIP)
print("Pipeline state:", PIPELINE_STATE_PATH)
for stage_name, run_dir in (("E2", E2_RUN_DIR), ("E3", E3_RUN_DIR)):
    for filename in (
        "pvuad_best_model.pth", "pvuad_last_model.pth",
        "pvuad_last_checkpoint.pth", "e2_teacher_model_fp16.pth",
    ):
        path = run_dir / filename
        if path.is_file():
            print(
                f"{stage_name:2s} {filename:30s} "
                f"{path.stat().st_size / 2**20:,.1f} MiB — {path}"
            )

if state.get("status") in {"e2_paused", "e3_paused", "e2_completed_e3_pending", "e3_complete_tta_pending"}:
    print(
        "\nDỪNG AN TOÀN: vì lượt chạy đã được khởi động bằng Save & Run All, "
        "Kaggle sẽ tự lưu Output của version này. Sau đó tạo session mới, "
        "gắn Output đó làm Input và tiếp tục bằng Save & Run All."
    )
elif state.get("status") == "completed":
    print("\nHOÀN TẤT E2 -> E3 + TTA. Không cần chạy thêm session.")


Pipeline status: completed
Active stage   : DONE
Next action    : None
TransReID initialization audit:


,checkpoint,loaded_tensor_count,target_tensor_count,mismatch_count,resized_pos_embed,source_sie_loaded,source_jpm_local_branches_loaded,global_jpm_branch_used_for_target_final_block,global_branch_mapped_tensor_count,embedding_dim,...,feature_shape,init_qkv_max_abs,global_final_qkv_max_abs,qkv_grad_norm,patch_grad_norm,backbone_all_trainable,source_sie_used,source_jpm_local_branches_used,source_jpm_global_final_branch_used,target_stride
0,/kaggle/working/vit_transreid_msmt.pth,152,152,0,"{'source_shape': [1, 211, 768], 'target_shape'...",False,False,True,14,768,...,"[3, 768]",0.0,0.0,0.004051,0.019206,True,False,False,True,"[16, 16]"


,method,epoch,mAP,Rank-1,Rank-5,Rank-10
0,UAD paper,NaN,11.000,29.500,46.300,54.800
1,E2 TransReID-init,60.0,11.665,30.773,47.291,55.816
2,E3-TDH TransReID-init,25.0,11.805,30.851,47.338,56.097
3,E3-TDH TransReID-init flip TTA,25.0,11.877,30.788,47.588,56.112



Report ZIP: /kaggle/working/pvuad_e2_e3_transreid_init_trainable_reports.zip
Pipeline state: /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/pipeline_state.json
E3 pvuad_best_model.pth           334.7 MiB — /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh/pvuad_best_model.pth
E3 pvuad_last_model.pth           334.7 MiB — /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh/pvuad_last_model.pth
E3 pvuad_last_checkpoint.pth      676.8 MiB — /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh/pvuad_last_checkpoint.pth
E3 e2_teacher_model_fp16.pth      167.4 MiB — /kaggle/working/pvuad_runs/e2_e3_transreid_init_trainable768_b24_v1/e3_tdh/e2_teacher_model_fp16.pth

HOÀN TẤT E2 -> E3 + TTA. Không cần chạy thêm session.


## Cách chạy qua nhiều phiên Kaggle

1. **Phiên đầu:** chọn **GPU T4 ×2**, attach `whu-mars`. Bật Internet để notebook tải `vit_transreid_msmt.pth`, hoặc attach checkpoint đó như một Kaggle Dataset.
2. **Không attach output E2/E3 cũ để resume** trong phiên đầu. Đây là pipeline tag mới.
3. Chạy bằng **Save Version → Save & Run All**. Trước E2 phải thấy dòng `TRANSREID_TRAINABLE_INIT_SMOKE_OK`.
4. Nếu notebook dừng sạch với `e2_paused`: attach **Output của đúng version vừa chạy** vào phiên kế tiếp rồi Save & Run All.
5. Khi E2 hoàn tất, notebook cố ý dừng ở `e2_completed_e3_pending`; attach output đó sang session mới để E3 bắt đầu.
6. E3 cũng có thể `e3_paused` và resume tương tự.
7. Khi thấy `completed`, lấy report ZIP và checkpoint E3 tốt nhất.

### Dấu hiệu run đúng

Ở fresh E2 log phải thể hiện:
- `MODEL.PRETRAIN_CHOICE = transreid`;
- feature vẫn **768-D**;
- target stride `[16,16]`;
- `source_sie_used=False`, `source_jpm_local_branches_used=False`, nhưng global `b1` được map vào block cuối;
- backbone **TRAINABLE**, không có `extractor_frozen=True`.

Ở resume/E3/TTA, `PRETRAIN_CHOICE=none` là **đúng** vì model lúc đó phải được phục hồi từ checkpoint WHU-MARS, không transplant source weight lần nữa.
